# Détection des maladies de la feuille de tomate — Entraînement

**Projet universitaire de Deep Learning** — MobileNetV2, Transfer Learning, Flask.

Ce notebook accompagne les scripts du dossier `training/` : il ne les remplace pas, il
montre les étapes et permet de vérifier rapidement le dataset, les métriques et une
prédiction.

Ordre d'exécution des scripts :

```bash
python training/download_dataset.py
python training/prepare_data.py
python training/train.py --epochs 15 --fine-tune-epochs 8
python training/evaluate.py
python app.py          # application Web (inférence temps réel)
```

## 1. Vérifier le dataset préparé

In [ ]:
import sys
from pathlib import Path

# Importer depuis la racine du projet, même si le notebook est lancé dans notebooks/
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))

import config

print("Classes :", config.CLASS_NAMES)
print("Taille d'image :", config.IMAGE_SIZE)

for split in ("train", "validation", "test"):
    dossier = config.PROCESSED_DATA_DIR / split
    etat = f"{len(list(dossier.iterdir()))} classes" if dossier.exists() else "absent"
    print(f"{split:>10} : {etat}")

## 2. Entraîner

L'entraînement est fait par le script (deux phases : base gelée puis fine-tuning) :

```bash
python training/train.py --epochs 15 --fine-tune-epochs 8
```

Les résultats officiels du rapport viennent de cette commande.

## 3. Lire les métriques sur l'ensemble de test

In [ ]:
from utils.helpers import charger_json

metriques = charger_json(config.METRICS_DIR / "evaluation.json")

if metriques:
    global_ = metriques["global"]
    print(f"Accuracy  : {global_['accuracy'] * 100:.2f} %")
    print(f"F1 macro  : {global_['f1_macro'] * 100:.2f} %")
    print(f"Erreurs   : {global_['nb_erreurs']} / {global_['nb_images']}")
else:
    print("Aucune métrique : lancer d'abord python training/evaluate.py")

## 4. Prédire une image

In [ ]:
from utils.helpers import nom_affichage
from utils.predictor import obtenir_predictor

predictor = obtenir_predictor()
predictor.charger()

images = sorted((config.PROCESSED_DATA_DIR / "test").glob("**/*.jpg"))

if images:
    resultat = predictor.predire(images[0])
    print("Image :", images[0].name)
    print("Classe prédite :", nom_affichage(resultat["classe"]))
    print(f"Confiance : {resultat['confiance'] * 100:.1f} %")
    print(f"Durée : {resultat['duree_ms']} ms")
else:
    print("Aucune image de test trouvée.")

## 5. Application Web

```bash
python app.py
```

Puis ouvrir <http://127.0.0.1:5000> : téléversement d'une image, prédiction, résultat.
L'application ne conserve ni image ni résultat.